<a href="https://www.kaggle.com/code/toivohaapasaari/catboostensemble?scriptVersionId=322223096" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

df=pd.read_csv("/kaggle/input/home-data-for-ml-course/train.csv").drop(columns=["Id"])

df.head()

## Feature Engineering etc

In [ ]:
#temporal features

df["Age"] = df["YrSold"] - df["YearBuilt"]
df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]

df = df.drop(columns=["YrSold","YearBuilt", "YearRemodAdd", "GarageYrBlt"])


#log-scaling target for linear performance.

y = np.log1p(df["SalePrice"])
df = df.drop(columns = ["SalePrice"]) #drop y. 

log_features = [
    "LotArea",
    "LotFrontage",
    "MasVnrArea",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "GrLivArea",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "PoolArea",
    "MiscVal"
]

for feat in log_features:
    df[feat] = np.log1p(df[feat])


standard_scale_features = [
    # engineered age variables
    "Age",
    "RemodAge",
    "GarageAge",

    # basic count variables
    "OverallQual",
    "OverallCond",
    "BsmtFullBath",
    "BsmtHalfBath",
    "FullBath",
    "HalfBath",
    "TotRmsAbvGrd",
    "Fireplaces",
    "GarageCars",

    # other numeric features
    "GarageArea",
    "MoSold",
]

#To prevent leakage,
#we split the data before scaling 
#since standard scaler takes the z of also the test data.

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.1,random_state = 777)

scaler = StandardScaler()

scaler.fit(X_train[standard_scale_features])

X_train[standard_scale_features]=scaler.transform(X_train[standard_scale_features])
X_test[standard_scale_features]=scaler.transform(X_test[standard_scale_features])


## Linear model

In [ ]:
numeric_features = [
    "LotFrontage",
    "LotArea",
    "OverallQual",
    "OverallCond",
    "MasVnrArea",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "GrLivArea",
    "BsmtFullBath",
    "BsmtHalfBath",
    "FullBath",
    "HalfBath",
    "BedroomAbvGr",
    "KitchenAbvGr",
    "TotRmsAbvGrd",
    "Fireplaces",
    "GarageCars",
    "GarageArea",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "PoolArea",
    "MiscVal",
    "MoSold",
    "Age",
    "RemodAge",
    "GarageAge"
]


X_tr = X_train[numeric_features].interpolate(method='linear')
X_ts = X_test[numeric_features].interpolate(method='linear')


import sklearn.linear_model as LinReg


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))
    
Fit_search=pd.DataFrame({"Lasso":[],"Ridge":[],"ElasticNet":[],"Alpha":[],"l1_ratio":[]})

for alph in np.arange(0.05,10,0.05):

    Ridge_reg=LinReg.Ridge(alpha=alph)
    Lasso_reg=LinReg.Lasso(alpha=alph)
    
    Ridge_reg.fit(X_tr,y_train)
    Lasso_reg.fit(X_tr,y_train)

    y_r = Ridge_reg.predict(X_ts)
    y_true = y_test
    Ridge_RMSE = rmse(y_true, y_r)
    
    y_l = Lasso_reg.predict(X_ts)
    Lasso_RMSE = rmse(y_true, y_l)

    for ratio in np.arange(0.05,0.97,0.2):
        Elastic_reg=LinReg.ElasticNet(alpha=alph,l1_ratio=ratio)
        Elastic_reg.fit(X_tr,y_train)

        y_e = Elastic_reg.predict(X_ts)
        Elastic_RMSE = rmse(y_true, y_e)
    
        Fit_search.loc[len(Fit_search)] = [Lasso_RMSE,Ridge_RMSE,Elastic_RMSE,alph,ratio]

Fit_search.describe()

## CatBoost

In [ ]:


X_train[numeric_features]=X_train[numeric_features].interpolate(method='linear')
X_test[numeric_features]=X_test[numeric_features].interpolate(method='linear')

Linear_model = LinReg.Ridge(alpha=1)
Linear_model.fit(X_train[numeric_features],y_train)

y_cat_train = y_train-Linear_model.predict(X_train[numeric_features])
y_cat_test = y_test-Linear_model.predict(X_test[numeric_features])

X_train["Ridge"]=Linear_model.predict(X_train[numeric_features])
X_test["Ridge"]=Linear_model.predict(X_test[numeric_features])
if "Ridge" not in numeric_features:
    numeric_features.append("Ridge")


Cathegorical =list(set(list(X_train.columns)) - set(numeric_features)) 


X_train[Cathegorical] = X_train[Cathegorical].fillna('nan').astype(str)
X_test[Cathegorical] = X_test[Cathegorical].fillna('nan').astype(str)



X_train[numeric_features] = X_train[numeric_features].apply(pd.to_numeric, errors='coerce')
X_test[numeric_features] = X_test[numeric_features].apply(pd.to_numeric, errors='coerce')



from catboost import CatBoostRegressor, Pool

train_pool = Pool(X_train, y_cat_train,cat_features = Cathegorical,)
valid_pool = Pool(X_test, y_cat_test,cat_features = Cathegorical,)


model = CatBoostRegressor(
    iterations=5801,     
    learning_rate=0.015,   
    depth=4,             
    eval_metric='RMSE',  
    verbose=100          
)

model.fit(
    train_pool,
    eval_set = valid_pool,  
    plot = False            
)



history = model.get_evals_result()

plt.plot(history['learn']['RMSE'], label='train')
plt.plot(history['validation']['RMSE'], label='validation')
plt.xlabel('Iteration')
plt.ylabel('RMSE')
plt.legend()
plt.show()




In [ ]:
test = pd.read_csv(r"/kaggle/input/home-data-for-ml-course/test.csv")

def prep(df, scaler=None,linear_model=Linear_model,cat= Cathegorical,model=model):

    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]
    df = df.drop(columns=["YrSold","YearBuilt", "YearRemodAdd", "GarageYrBlt"])

    log_features = ["LotArea","LotFrontage","MasVnrArea","BsmtFinSF1","BsmtFinSF2","BsmtUnfSF",
        "TotalBsmtSF","1stFlrSF","2ndFlrSF","LowQualFinSF","GrLivArea","WoodDeckSF",
        "OpenPorchSF","EnclosedPorch","3SsnPorch","ScreenPorch","PoolArea","MiscVal"]
    for feat in log_features:
        df[feat] = np.log1p(df[feat])

    standard_scale_features = ["Age","RemodAge","GarageAge","OverallQual","OverallCond",
        "BsmtFullBath","BsmtHalfBath","FullBath","HalfBath","TotRmsAbvGrd",
        "Fireplaces","GarageCars","GarageArea","MoSold"]
    if scaler is None:
        scaler = StandardScaler()
        df[standard_scale_features]=scaler.fit_transform(df[standard_scale_features])
    else: 
        df[standard_scale_features]=scaler.transform(df[standard_scale_features])
    
    df[Cathegorical] = df[Cathegorical].fillna('nan').astype(str)
    #ridge_features = [f for f in numeric_features if f != "Ridge"]
    #df[ridge_features] = df[ridge_features].interpolate(method='linear')
    #df["Ridge"] = linear_model.predict(df[ridge_features])
    ridge_features = [f for f in numeric_features if f != "Ridge"]
    #print(set(X_train[ridge_features].columns) - set(df[ridge_features].columns))
    df[ridge_features] = df[ridge_features].interpolate(method='linear')
    df["Ridge"] = linear_model.predict(df[ridge_features].values)

    return df


test_transformed = prep(test, scaler, Linear_model)
test_transformed = test_transformed[X_train.columns]
test_pool = Pool(test_transformed, cat_features=Cathegorical)

ridge_features = [f for f in numeric_features if f != "Ridge"]

pred = np.expm1(Linear_model.predict(test_transformed[ridge_features]) + model.predict(test_pool))

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": pred
}).set_index("Id")

In [ ]:
submission.to_csv("submission.csv")
submission